In [ ]:
#TODO consider removing.


In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from its.search import InverseTransformationSearch
from search.gradient_descent import ParallelGradientDescent

from src.utils.sampling import BatchNegativeSampler

#torch.cuda.is_available = lambda: False
#device = torch.device("cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "bigger_emnist"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}



architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:
from src.utils.transforms.apply import grid_resample_border,grid_resample_reflection

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
x=next(iter(test_loader_transformed))[0]

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]


In [ ]:
from src.utils.eval.vis import vis_dataset

vis_dataset(train_loader,val_loader,test_loader_transformed)

In [ ]:
from model.train import train_and_get_model,train_or_load_energy_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    model_dir_path = os.path.join(current_path, "experiment_files", "models")
    embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
    # Add results dir and helper for save paths
    results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods")
    os.makedirs(results_dir_path, exist_ok=True)

    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
model.train()

In [ ]:
#chek if data is image data
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

In [ ]:
from src.utils.replacer import replace_rotation_transforms_2vec

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]





from src.utils.transforms.apply import grid_resample
from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:
transform_seq.transformations

In [ ]:
from model.get_model import get_network_layer

layer,layer_io = get_network_layer(dataset_info, architecture, 0, num_classes=None, num_rotations=8)

In [ ]:
from confidence.direct.logit_based import EnergyConfidence
from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence

logit_energy = SinglePassConfidence(model, EnergyConfidence(), index=None)
problem_energy_logits = TransformationProblem(logit_energy, transform_seq,                                              consolidate_method="consolidate_simple")
#test ot
from search.random_search import RSLR
optimizer = RSLR(initial_samples=46, local_runs=2, local_max_steps=3,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer = RSLR(initial_samples=60,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})


optimizer_120 = RSLR(initial_samples=120-27, local_runs=3, local_max_steps=4,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer_120 = RSLR(initial_samples=120,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})

optimizer_240 = RSLR(initial_samples=195, local_runs=5, local_max_steps=4,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer_240 = RSLR(initial_samples=240,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})

optimizer_960 = RSLR(initial_samples=960-45, local_runs=5, local_max_steps=4,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer_960 = RSLR(initial_samples=960,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})


from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search, evaluate_confidence_and_search, \
    ITSWRAPPER


In [ ]:
#use torchinfo to measure flops
import torchinfo
input = next(iter(test_loader_transformed))[0].to(device)
model_half = get_network(dataset_info,architecture+"_half", num_classes=1).to(device)
model_quarter = get_network(dataset_info,architecture+"_quarter", num_classes=1).to(device)





In [ ]:
#speed test main model at 1 times batch
#half at 4 times batch and quarter at 16 times batch

import time
def time_model(model, x, iterations=100):
    #warm up
    _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for i in range(iterations):
        _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()
    avg_time = (end - start) / iterations
    return avg_time






In [ ]:
x = next(iter(test_loader_transformed))[0].to(device)
#half base x
x = x[:max(batch_size//2,1)]

In [ ]:
#x shape may vary so repeat first
x_4 = x.repeat_interleave(4, dim=0)
x_16 = x.repeat_interleave(16, dim=0)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#time_main = time_model(model, x, iterations=500)
#time_half = time_model(model_half, x_4,iterations=500)
#time_quarter = time_model(model_quarter, x_16, iterations=500)
#print(f"Main model time: {time_main:.3f}s")
#print(f"Half model time (4x batch): {time_half:.3f}s")
#print(f"Quarter model time (16x batch): {time_quarter:.3f}s")

In [ ]:
del model_half
del model_quarter

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("energy_confidence_transformed"), show_progress=True,
    repeats=5)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.utils.augments import ComposeAugmentations, random_gaussian_noise, random_contrast, \
    random_gamma  ,random_blur_or_sharpen,build_default_augmentations
import src.utils.augments

def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from src.utils.augments import build_default_augmentations, small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling_strategy
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler


energy_model2 = get_network(dataset_info,architecture, num_classes=1).to(device)



if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None



negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
namef = f"{modelname}_energy2" if not is_image_data else f"{modelname}_energy2_no_augment_energy2_only_blur_aug"
energy_conf_default = train_or_load_energy_model(
    energy_model2, model_dir_path, namef, train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_default.to(device).eval()



problem_energy_default = TransformationProblem(energy_conf_default, transform_seq,
                                               consolidate_method="consolidate_simple")
savepathname = savepath("learned_energy_confidence_only_blur_aug_transformed") if is_image_data else savepath("learned_energy_confidence_transformed")
res = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_default,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepathname, show_progress=True,
    repeats=5)

#unload values
del energy_model2
del energy_conf_default
del problem_energy_default
print(res)


In [ ]:
architecture_quarter = architecture+"_quarter"


In [ ]:
test_loader_transformed_quarter = torch.utils.data.DataLoader(
    test_loader_transformed.dataset,
    batch_size=max(dataset_info.batch_size//4,1),
    shuffle=False,
    num_workers=test_loader_transformed.num_workers,
    pin_memory=test_loader_transformed.pin_memory,
    persistent_workers=test_loader_transformed.persistent_workers
)
test_loader_transformed_sixteenth = torch.utils.data.DataLoader(
    test_loader_transformed.dataset,
    batch_size=max(dataset_info.batch_size//16,1),
    shuffle=False,
    num_workers=test_loader_transformed.num_workers,
    pin_memory=test_loader_transformed.pin_memory,
    persistent_workers=test_loader_transformed.persistent_workers
)

In [ ]:

architecture_half = architecture+"_half"

energy_model2_half = get_network(dataset_info,architecture_half, num_classes=1).to(device)

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    = transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_half = train_or_load_energy_model(
    energy_model2_half, model_dir_path, f"{modelname}_energy2_half", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_half.to(device).eval()
problem_energy_half = TransformationProblem(energy_conf_half, transform_seq,                                              consolidate_method="consolidate_simple")
res = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_half,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_half_transformed"), show_progress=True,
    repeats=5)

res_120 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_half_transformed_120"), show_progress=True,
    repeats=5)

res_240 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_half,
    test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_half_transformed_240"), show_progress=True,
    repeats=5)


#optimizer_120_2_tu_berlin = RSLR(initial_samples=120-20,#local_runs=2, local_max_steps=3,local_opt_kwargs={"lr":0.1})
#energy_model2_half.train()
#problem_energy_half.confidence_module.train()

#res_120_2 = load_or_run_evaluate_confidence_and_search(
#    model, optimizer=optimizer_120_2_tu_berlin,
#    problem=problem_energy_half,
#    #test_loader=test_loader_transformed_quarter,
#    max_batch_override=dataset_info.batch_size_search,
#    save_path=savepath
#    ("learned_energy_confidence_half_transformed_120_tu_berlin"),
#    show_progress=True,
#    repeats=5)
#model.eval()
#energy_model2_half.eval()

#unload values
del energy_model2_half
del energy_conf_half
del problem_energy_half
print(res["accuracy_mean"])
print(res_120["accuracy_mean"])
print(res_240["accuracy_mean"])




In [ ]:
#freeze main model
for param in model.parameters():
    param.requires_grad = False


architecture_half = architecture+"_half"

energy_model2_half = get_network(dataset_info,architecture_half, num_classes=1).to(device)

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    = transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_half = train_or_load_energy_model(
    energy_model2_half, model_dir_path, f"{modelname}_energy2_half_longer", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True,deterministic_val=True)

energy_conf_half.to(device).eval()
problem_energy_half = TransformationProblem(energy_conf_half, transform_seq,                                              consolidate_method="consolidate_simple")

res_120 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_half_transformed_120_longer"), show_progress=True,
    repeats=5)

#unload values
del energy_model2_half
del energy_conf_half
del problem_energy_half

print(res_120["accuracy_mean"])





In [ ]:
energy_model2_quarter = get_network(dataset_info,architecture_quarter, num_classes=1).to(device)

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    = transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_quarter = train_or_load_energy_model(
    energy_model2_quarter, model_dir_path, f"{modelname}_energy2_quarter", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)
energy_conf_quarter.to(device).eval()
problem_energy_quarter = TransformationProblem(energy_conf_quarter, transform_seq,                                              consolidate_method="consolidate_simple")


res = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_quarter,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_quarter_transformed"), show_progress=True,
    repeats=5)

res_120 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_quarter,
    test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_quarter_transformed_120"), show_progress=True,
    repeats=5)
res_240 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_quarter,
    test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_quarter_transformed_240"), show_progress=True,
    repeats=5)
res_960 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_960, problem=problem_energy_quarter,
    test_loader=test_loader_transformed_sixteenth, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_quarter_transformed_960"), show_progress=True,
    repeats=5)
#unload values
del energy_model2_quarter
del energy_conf_quarter
del problem_energy_quarter
print(res)
print(res_120)
print(res_240)
print(res_960)

In [ ]:
#now no dec strat quarter
if True:
    
    negative_sampling_module = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, ), transform_true_function
        = transform_true_function, augment_function=affine_augment,
        decision_strategy=None,
    )
    energy_model2_quarter = get_network(dataset_info,architecture_quarter, num_classes=1).to(device)

    energy_conf_quarter = train_or_load_energy_model(
        energy_model2_quarter, model_dir_path, f"{modelname}_energy2_quarter_no_dec_strat", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module, load_if_exists=True)
    energy_conf_quarter.to(device).eval()
    problem_energy_quarter = TransformationProblem(energy_conf_quarter, transform_seq,
                                                   consolidate_method="consolidate_simple")
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_quarter,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_quarter_transformed_no_dec_strat"), show_progress=True,
        repeats=5)
    res_120 = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer_120, problem=problem_energy_quarter,
        test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_quarter_transformed_no_dec_strat_120"), show_progress=True,
        repeats=5)
    res_240 = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer_240, problem=problem_energy_quarter,
        test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_quarter_transformed_no_dec_strat_240"), show_progress=True,
        repeats=5)
    res_960 = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer_960, problem=problem_energy_quarter,
        test_loader=test_loader_transformed_sixteenth, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_quarter_transformed_no_dec_strat_960"), show_progress=True,
        repeats=5)
    # unload values
    del energy_conf_quarter
    del problem_energy_quarter
    print(res)
    print(res_120)
    print(res_240)
    print(res_960)





In [ ]:
#now half no dec strat
if True:

    
    energy_model2_half = get_network(dataset_info,architecture_half, num_classes=1).to(device)
    negative_sampling_module = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, ), transform_true_function
        = transform_true_function, augment_function=affine_augment,
        decision_strategy=None,
    )
    energy_conf_half = train_or_load_energy_model(
        energy_model2_half, model_dir_path, f"{modelname}_energy2_half_no_dec_strat", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module, load_if_exists=True)
    energy_conf_half.to(device).eval()
    problem_energy_half = TransformationProblem(energy_conf_half, transform_seq,
                                                consolidate_method="consolidate_simple")
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_half,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_half_transformed_no_dec_strat"), show_progress=True,
        repeats=5)
    res_120 = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer_120, problem=problem_energy_half,
        test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_half_transformed_no_dec_strat_120"), show_progress=True,
        repeats=5)
    res_240 = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer_240, problem=problem_energy_half,
        test_loader=test_loader_transformed_quarter, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_half_transformed_no_dec_strat_240"), show_progress=True,
        repeats=5)
    # unload values
    del energy_conf_half
    del problem_energy_half
    print(res)
    print(res_120)
    print(res_240)

In [ ]:
from src.utils.eval.vis import plt_setup_latex

#plot the results
W= plt_setup_latex()
def load_results(label: str):
    import json
    path = savepath(label)
    with open(path, "r") as f:
        data = json.load(f)
    return data

In [ ]:
figure_path = os.path.join(current_path,"experiment_files","export","fig","comparision_supervised_lb",dataset,transform_name)
os.makedirs(figure_path, exist_ok=True)

In [ ]:
from matplotlib import ticker
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Define Constants and Setup (Assuming W and figure_path are defined) ---

budgets = [60, 120, 240, 960]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"] # Color for each budget



# --- Experiment Definitions and Grouping ---
model_based_experiments = ["Full", "Half", "Quarter"]
data_based_experiments = ["Full Data-based", "Half Data-based", "Quarter Data-based"]
plotting_experiments = model_based_experiments + data_based_experiments

# --- Organized Data Mapping (Remains the same as previous) ---
exp_to_budget_data = {
    "Full": {"60": load_results("learned_energy_confidence_transformed")},
    "Half": {
        "60": load_results("learned_energy_confidence_half_transformed"),
        "120": load_results("learned_energy_confidence_half_transformed_120"),
        "240": load_results("learned_energy_confidence_half_transformed_240")
    },
    "Quarter": {
        "60": load_results("learned_energy_confidence_quarter_transformed"),
        "120": load_results("learned_energy_confidence_quarter_transformed_120"),
        "240": load_results("learned_energy_confidence_quarter_transformed_240"),
        "960": load_results("learned_energy_confidence_quarter_transformed_960")
    },
    "Full Data-based": {"60": load_results("learned_energy_confidence_no_dec_strat_transformed")},
    "Half Data-based": {
        "60": load_results("learned_energy_confidence_half_transformed_no_dec_strat"),
        "120": load_results("learned_energy_confidence_half_transformed_no_dec_strat_120"),
        "240": load_results("learned_energy_confidence_half_transformed_no_dec_strat_240")
    },
    "Quarter Data-based": {
        "60": load_results("learned_energy_confidence_quarter_transformed_no_dec_strat"),
        "120": load_results("learned_energy_confidence_quarter_transformed_no_dec_strat_120"),
        "240": load_results("learned_energy_confidence_quarter_transformed_no_dec_strat_240"),
        "960": load_results("learned_energy_confidence_quarter_transformed_no_dec_strat_960")
    }
}


# --- Prepare data for plotting ---

width = 0.2  # width of each bar
num_model_based = len(model_based_experiments)

# 1. Calculate adjusted X positions for separation
x_ticks_adjusted = []
current_x = 0
separator_pos = num_model_based # Index where the Data-based group starts

for i in range(len(plotting_experiments)):
    if i == separator_pos:
        current_x += 0.5 # The extra space is 1 unit wide
    x_ticks_adjusted.append(current_x)
    current_x += 1
x_ticks_adjusted = np.array(x_ticks_adjusted)

# --- Plotting ---

# Assuming W and figure_path are defined globally
# W = 10
# figure_path = "."
# if not os.path.exists(figure_path): os.makedirs(figure_path)

plt.figure(figsize=(W/2, W / 3.5),dpi=300)
plt.grid(axis="y", linestyle=":", alpha=0.8,zorder=-1)

bars = []

# Loop through each budget to plot a set of bars
for i, budget in enumerate(budgets):
    means = []
    ses = []

    # Collect data for the current budget across all experiments in plotting order
    for exp in plotting_experiments:
        res = exp_to_budget_data.get(exp, {}).get(str(budget), None)
        if res:
            means.append(res["accuracy_mean"])
            ses.append(res["accuracy_se"] if res["accuracy_se"] is not None else 0)
        else:
            means.append(np.nan)
            ses.append(0)

    ses = np.array(ses) * 1.96  # 95% CI (1.96 * SE)

    # Calculate the shift for the current budget group
    shift = (i - (len(budgets) - 1) / 2) * width

    # Plot bars
    bar = plt.bar(
        x_ticks_adjusted + shift,
        means,
        width,
        label=f"{budget} budget",
        yerr=ses,
        capsize=1,
        color=colors[i],
        edgecolor="black",
        linewidth = 0.2,zorder=3,
        #reduce error bar width
        error_kw=dict(lw=0.5,capthick=0.5),


    )
    bars.append(bar)

# --- Style Plot ---

#Define the labels for the x-axis
x_labels = [exp.replace(' Data-based', '') for exp in plotting_experiments]

#Set the x-ticks, moving them significantly further down
plt.xticks(x_ticks_adjusted, x_labels, rotation=25, ha='right', y=-0.12,fontsize=8)
plt.yticks(fontsize=8)


vertical_line_x_pos = x_ticks_adjusted[separator_pos] - 0.6 #
vertical_line_x_pos = (x_ticks_adjusted[separator_pos - 1] + x_ticks_adjusted[separator_pos]) / 2

plt.axvline(x=vertical_line_x_pos, color='gray', linestyle='--', linewidth=1)

#Adjust group headers' Y position further down to account for the lowered x-labels
model_group_center = x_ticks_adjusted[:num_model_based].mean()
data_group_center = x_ticks_adjusted[num_model_based:].mean()
ax = plt.gca()  # Get current axes

# transform=ax.get_xaxis_transform() keeps x in data coords, y in axes fraction
ax.text(model_group_center, -0.17, "Model-based",
        ha='center', fontsize=8, weight='bold',
        transform=ax.get_xaxis_transform())
ax.text(data_group_center, -0.17, "Data-based",
        ha='center', fontsize=8, weight='bold',
        transform=ax.get_xaxis_transform())
plt.ylabel("Accuracy", fontsize=9)
#plt.legend(title="Optimization Budget", fontsize=8, loc='upper left')

all_means = []
all_upper_bounds = []

for budget in budgets:
    for exp in plotting_experiments:
        res = exp_to_budget_data.get(exp, {}).get(str(budget), None)
        if res:
            mean = res["accuracy_mean"]
            se = res["accuracy_se"] if res["accuracy_se"] is not None else 0
            ci = 1.96 * se
            all_means.append(mean)
            all_upper_bounds.append(mean + ci)

# Determine the min and max with a margin
y_min = max(0, min(all_means) - 0.05)  # Lower bound can't go below 0
y_max = min(1, max(all_upper_bounds) + 0.05)  # Upper bound can't go above 1

plt.ylim(y_min, y_max)
ax = plt.gca()  # Get current axes
tick_spacing = 0.05  # e.g., ticks every 0.1
ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=7))
plt.subplots_adjust(bottom=0.30)





plt.savefig(os.path.join(figure_path, "comparision_supervised_methods_lower_budget.pdf"), bbox_inches='tight')
plt.savefig(os.path.join(figure_path, "comparision_supervised_methods_lower_budget.pgf"), bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
exp_to_budget_data

In [ ]:
# --- Create a separate figure for legend ---
fig_legend = plt.figure(figsize=(W/2, W/2.5))  # width x height
ax = fig_legend.add_subplot(111)
ax.axis('off')  # no axes


legend = ax.legend(
    [b[0] for b in bars],  # first patch of each bar container
    [f"{budget} budget" for budget in budgets],
    loc='center',
    ncol=1,
    frameon=False, fontsize=10,
    title="Optimization Budget",
)
fig_legend.savefig("legend_only.pdf", bbox_inches='tight')
fig_legend.savefig("legend_only.pgf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Speed testing function
def time_model_forward(model, x, iterations=100, warmup=10):
    """Time the forward pass of a model."""
    model.eval()
    with torch.no_grad():
        # Warmup
        for _ in range(warmup):
            _ = model(x)

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(iterations):
            _ = model(x)

        if device.type == "cuda":
            torch.cuda.synchronize()

        end = time.perf_counter()

    avg_time = (end - start) / iterations
    return avg_time


mult = 4

# Prepare test batches
x_full = next(iter(test_loader_transformed))[0].to(device)
x_half = x_full.repeat_interleave(2 * mult, dim=0)
x_quarter = x_full.repeat_interleave(4 * mult, dim=0)
x_full = x_full.repeat_interleave(1 * mult, dim=0)


speed_results = {}

# ------------------------------
# Full model
# ------------------------------
print("Testing full model...")

full_batch_time = time_model_forward(model, x_full)
speed_results['full'] = {
    'batch_size': x_full.shape[0],
    'time_per_batch': full_batch_time,
    'time_per_sample': full_batch_time / x_full.shape[0]
}

# ------------------------------
# Half model
# ------------------------------
print("Testing half model...")

model_half = get_network(dataset_info, architecture + "_half", num_classes=1).to(device)
model_half_path = os.path.join(model_dir_path, f"{modelname}_energy2_half.ckpt")

if os.path.exists(model_half_path):
    checkpoint = torch.load(model_half_path, map_location=device)
    model_half.load_state_dict(checkpoint.get('state_dict', checkpoint))
model_half.eval()

half_batch_time = time_model_forward(model_half, x_half)
half_4x_time = time_model_forward(model_half, x_full[:min(x_half.shape[0] * 4, x_full.shape[0])])

speed_results['half'] = {
    'batch_size': x_half.shape[0],
    'time_per_batch': half_batch_time,
    'time_per_sample': half_batch_time / x_half.shape[0],
    'time_4x_batch': half_4x_time
}

# ------------------------------
# Quarter model
# ------------------------------
print("Testing quarter model...")

model_quarter = get_network(dataset_info, architecture + "_quarter", num_classes=1).to(device)
model_quarter_path = os.path.join(model_dir_path, f"{modelname}_energy2_quarter.ckpt")

if os.path.exists(model_quarter_path):
    checkpoint = torch.load(model_quarter_path, map_location=device)
    model_quarter.load_state_dict(checkpoint.get('state_dict', checkpoint))
model_quarter.eval()

quarter_batch_time = time_model_forward(model_quarter, x_quarter)
quarter_16x_time = time_model_forward(
    model_quarter, x_full[:min(x_quarter.shape[0] * 16, x_full.shape[0])]
)

speed_results['quarter'] = {
    'batch_size': x_quarter.shape[0],
    'time_per_batch': quarter_batch_time,
    'time_per_sample': quarter_batch_time / x_quarter.shape[0],
    'time_16x_batch': quarter_16x_time
}

# Save
speed_results_path = os.path.join(results_dir_path, transform_name, "speed_test_results.json")
os.makedirs(os.path.dirname(speed_results_path), exist_ok=True)

with open(speed_results_path, 'w') as f:
    json.dump(speed_results, f, indent=2)

print(f"Speed test results saved to {speed_results_path}")


In [ ]:
# Plot the speed test results
from src.utils.eval.vis import plt_setup_latex

W = plt_setup_latex()

# Load results
with open(speed_results_path, 'r') as f:
    speed_results = json.load(f)

# Create comparison plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(W, W/2))

# Plot 1: Time per sample
models = ['full', 'half', 'quarter']
times_per_sample = [speed_results[m]['time_per_sample'] * 1000 for m in models]  # Convert to ms
colors_plot = ['#1f77b4', '#ff7f0e', '#2ca02c']

ax1.bar(models, times_per_sample, color=colors_plot, edgecolor='black')
ax1.set_ylabel('Time per sample (ms)')
ax1.set_xlabel('Model size')
ax1.set_title('Forward pass time per sample')
ax1.grid(axis='y', linestyle='--', alpha=1)

# Plot 2: Relative overhead
relative_times = [speed_results[m]['time_per_sample'] / speed_results['full']['time_per_sample']
                  for m in models]

ax2.bar(models, relative_times, color=colors_plot, edgecolor='black')
ax2.axhline(y=1.0, color='red', linestyle='--', linewidth=1, label='Full model baseline')
ax2.set_ylabel('Relative time (normalized to full)')
ax2.set_xlabel('Model size')
ax2.set_title('Relative forward pass overhead')
ax2.legend()
ax2.grid(axis='y', linestyle='--', alpha=1)


plt.savefig(os.path.join(figure_path, "model_speed_comparison.pdf"), bbox_inches='tight')
plt.savefig(os.path.join(figure_path, "model_speed_comparison.pgf"), bbox_inches='tight', dpi=300)
plt.show()

# Print summary
print("\nSpeed Test Summary:")
print("-" * 50)
for model_name in models:
    print(f"{model_name.capitalize()} model:")
    print(f"  Time per sample: {speed_results[model_name]['time_per_sample']*1000:.4f} ms")
    print(f"  Time per batch: {speed_results[model_name]['time_per_batch']*1000:.4f} ms")
    print(f"  Batch size: {speed_results[model_name]['batch_size']}")
    if model_name != 'full':
        speedup = speed_results['full']['time_per_sample'] / speed_results[model_name]['time_per_sample']
        print(f"  Speedup vs full: {speedup:.2f}x")
    print()

#save the speedup as json
speedup_results = {
    model_name: {
        'speedup_vs_full': speed_results['full']['time_per_sample'] / speed_results[model_name]['time_per_sample']
    } for model_name in models if model_name != 'full'
}
with open(os.path.join(figure_path, "model_speedup_results.json"), 'w') as f:
    json.dump(speedup_results, f, indent=2)

In [ ]:
def generate_multi_dataset_speed_table():
    """Generate LaTeX table comparing speedup across all datasets."""

    datasets = ["bigger_mnist", "bigger_emnist", "coil100", "tu_berlin", "modelnet10"]
    models = ['half', 'quarter']

    # Load results for all datasets
    all_speedups = {}
    for ds in datasets:
        # Get the correct transform_seq_name for this dataset
        ds_info = get_dataset_info(ds)

        results_path = os.path.join(
            current_path, "experiment_files", "results", ds,
            default_architecutre_mapping[ds], "comparison_supervised_methods",
            ds_info.transform_seq_name,
            "speed_test_results.json"
        )

        if os.path.exists(results_path):
            with open(results_path, 'r') as f:
                speed_results = json.load(f)
                all_speedups[ds] = {
                    model: speed_results['full']['time_per_sample'] / speed_results[model]['time_per_sample']
                    for model in models
                }
        else:
            print(f"Warning: Speed results not found for {ds} at {results_path}")

    # Generate LaTeX table
    latex = r"\begin{table}[h]" + "\n"
    latex += r"\centering" + "\n"
    latex += r"\begin{tabular}{lcc}" + "\n"
    latex += r"\toprule" + "\n"
    latex += r"Dataset & Half Model & Quarter Model \\" + "\n"
    latex += r"\midrule" + "\n"

    for ds in datasets:
        if ds in all_speedups:
            half_speedup = all_speedups[ds]['half']
            quarter_speedup = all_speedups[ds]['quarter']
            ds_display = ds.replace("_", " ").title()
            latex += f"{ds_display} & {half_speedup:.2f}$\\times$ & {quarter_speedup:.2f}$\\times$ \\\\\n"

    latex += r"\bottomrule" + "\n"
    latex += r"\end{tabular}" + "\n"
    latex += r"\caption{Speedup comparison across datasets (relative to full model).}" + "\n"
    latex += r"\label{tab:speedup_comparison}" + "\n"
    latex += r"\end{table}" + "\n"

    # Save to file
    table_path = os.path.join(figure_path, "speedup_multi_dataset_table.tex")
    with open(table_path, 'w') as f:
        f.write(latex)

    print(f"LaTeX table saved to {table_path}")
    print("\nGenerated LaTeX code:")
    print(latex)

    return latex
generate_multi_dataset_speed_table()